# Guardrails and untrusted content

## Northstar’s poisoned checkout runbook

This notebook teaches defense in depth for an incident agent that retrieves a runbook containing an indirect prompt injection. The default path is deterministic and performs no live tool calls.

**Outcomes:** map trust boundaries; quarantine suspicious material; delimit useful documents as data; validate typed actions at the tool boundary; and evaluate whether attacks can cause harm.


## Defense in depth

![Guardrail boundaries from untrusted input to constrained execution](assets/guardrail-boundaries.svg)

This static SVG is used because notebook Mermaid/JavaScript renderers are not reliable on GitHub. The README includes an editable Mermaid architecture diagram.


## 1. Injection is a trust-boundary problem

Direct injection is in a user request. Indirect injection arrives through a webpage, PDF, runbook, email, tool response, memory, or OCR text. The model may see either as tokens; your application must keep external content from gaining the authority of system policy or a user-approved action.

A detector can help, but no detector is perfect. Therefore model instructions, provenance classification, context isolation, structured output, tool validation, least privilege, human approval, sandboxing, and auditing each limit a different failure mode.


In [ ]:
from lab import (
    Document, POISONED_RUNBOOK, SAFE_RUNBOOK, classify_document, build_context,
    safe_investigation, adversarial_suite, ToolCall, validate_tool_call
)

poisoned = Document('checkout-runbook-v7', POISONED_RUNBOOK, 'northstar')
gate = classify_document(poisoned)
print('allowed?', gate.allowed)
print('reasons:', gate.reasons)
print(build_context(poisoned, gate))


## 2. Context isolation: useful data is still not authority

A clean-looking runbook is not a system instruction. Preserve provenance and delimit it in the model context; retrieve only tenant-authorized, task-relevant snippets. Never let document text directly select a tool, tenant, destination, or permission level.


In [ ]:
safe = Document('checkout-runbook-v6', SAFE_RUNBOOK, 'northstar')
safe_gate = classify_document(safe)
print(build_context(safe, safe_gate))
assert safe_gate.allowed
assert 'untrusted_document' in build_context(safe, safe_gate)


## 3. Tool guardrails are the enforcement point

The poisoned document asks for a restart. Even if a weak detector misses the text or a model repeats the request, the application validates the tool, typed arguments, tenant, and approval state. Unknown tools default deny; high-risk tools require application-owned approval.


In [ ]:
result = safe_investigation(poisoned)
print('document:', result['document_gate'])
print('requested tool:', result['proposed_call'])
print('tool gate:', result['tool_gate'])
assert not result['tool_gate'].allowed

cross_tenant = validate_tool_call(ToolCall('query_logs', {'tenant_id': 'globex'}, 'model'), 'northstar')
unknown = validate_tool_call(ToolCall('delete_records', {}, 'model'), 'northstar')
print('cross-tenant:', cross_tenant.reasons)
print('unknown tool:', unknown.reasons)


## 4. Experiment A — detector failure is not permission failure

For learning only, remove one marker from `INJECTION_MARKERS` in a scratch copy. The attacker’s text might then enter a delimited context, but the high-risk restart still cannot execute without human approval. This is the essential lesson: text filtering is probabilistic; authorization is deterministic.


## 5. Experiment B — adversarial release suite

A guardrail release test must check both *detection* and *harm prevention*. The suite covers poisoned content, benign-but-untrusted content, cross-tenant reads, and unknown tool invocation. Extend it with encoded, multilingual, split payload, image/OCR, stale-memory, and tool-response attacks.


In [ ]:
for i, case in enumerate(adversarial_suite(), 1):
    print(f'case {i}:', case)

# A useful release invariant: no unapproved high-risk tool reaches an executor.
assert not safe_investigation(poisoned)['tool_gate'].allowed


## 6. Production architecture and anti-patterns

**Do:** attribute source/tenant/freshness; isolate context; use strict schemas; keep tools narrow; apply egress and data filters; gate high-impact actions; use idempotency; redact logs; and continuously red-team complete trajectories.

**Do not:** rely on “ignore malicious instructions” in a prompt; grant a model broad admin or shell access; trust retrieval relevance as authorization; pass raw tool output into privileged context; or count a blocked phrase as proof that an attack cannot cause damage.

Optional framework integration: OpenAI Agents SDK and LangChain/LangGraph can offer input/output/tool guardrail hooks, but application-level authorization and tenant checks remain required.


## Exercises and references

1. Add an approved-domain egress guard for customer notifications.
2. Add a strict schema for a rollback proposal and reject extra fields.
3. Build a metric set: attack success, harmful action rate, false positives, false negatives, and time-to-containment.
4. Model a poisoned image caption and decide which stage classifies it.

- [OWASP LLM01 Prompt Injection](https://genai.owasp.org/llmrisk/llm01-prompt-injection/)
- [OpenAI agent safety guidance](https://developers.openai.com/api/docs/guides/agent-builder-safety)
- [LangChain guardrails](https://docs.langchain.com/oss/python/langchain/guardrails)
- [Indirect prompt injection research](https://arxiv.org/abs/2302.12173)
